In [1]:
!pip -q install scikit-learn xgboost

Cell 2 — Imports

In [2]:
import os
import pickle
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

from xgboost import XGBRegressor

Cell 3 — Create output folder

In [3]:
os.makedirs("data/12_round3_finbert", exist_ok=True)

Cell 4 — Load Round 1 improved dataset and updated FinBERT files

In [4]:
round1_df = pd.read_csv("round1_improved_baseline_dataset.csv")
aligned_df = pd.read_csv("dataset_v2_aligned_finbert_updated.csv")
raw_emb = np.load("finbert_embeddings_raw_updated.npy")

print("Round 1 shape:", round1_df.shape)
print("Aligned Step 7 shape:", aligned_df.shape)
print("Raw embeddings shape:", raw_emb.shape)
print("aligned_df rows:", len(aligned_df))
print("raw_emb rows:", raw_emb.shape[0])

assert len(aligned_df) == raw_emb.shape[0], "Embedding rows do not match aligned_df rows"

Round 1 shape: (2181, 41)
Aligned Step 7 shape: (2247, 26)
Raw embeddings shape: (2247, 768)
aligned_df rows: 2247
raw_emb rows: 2247


Cell 5 — Standardise merge keys

In [5]:
for df in [round1_df, aligned_df]:
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["filing_type"] = df["filing_type"].astype(str).str.strip().str.upper()
    df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce")

if "accession_number" in round1_df.columns:
    round1_df["accession_number"] = round1_df["accession_number"].astype(str).str.strip()

if "accession_number" in aligned_df.columns:
    aligned_df["accession_number"] = aligned_df["accession_number"].astype(str).str.strip()

Cell 6 — Add embedding index and merge into Round 1 dataset

In [6]:
aligned_df = aligned_df.copy()
aligned_df["emb_idx"] = np.arange(len(aligned_df))

merge_keys = ["ticker", "filing_date", "filing_type"]
if "accession_number" in round1_df.columns and "accession_number" in aligned_df.columns:
    merge_keys.append("accession_number")

round3_df = round1_df.merge(
    aligned_df[merge_keys + ["emb_idx"]],
    on=merge_keys,
    how="left"
)

print("Merged Round 3 shape:", round3_df.shape)
print("Missing emb_idx rows:", round3_df["emb_idx"].isna().sum())
round3_df.head()

Merged Round 3 shape: (2181, 42)
Missing emb_idx rows: 0


,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,...,mean_abs_return_5d,mean_abs_return_10d,max_abs_return_10d,return_skew_10d,return_kurtosis_10d,log_price_tminus1,vol_ratio_5_20,vol_ratio_10_60,log_future_realized_vol_10d,emb_idx
0,AAPL,320193,2019-05-01,10-Q,0000320193-19-000066,2019,2,320193,32019319000066,AAPL_20190501_10-Q_000032019319000066.txt,...,0.007233,0.007704,0.019473,-0.006040,0.587393,3.915367,0.870016,0.974208,-3.810629,1
1,AAPL,320193,2019-07-31,10-Q,0000320193-19-000076,2019,3,320193,32019319000076,AAPL_20190731_10-Q_000032019319000076.txt,...,0.005166,0.008841,0.022854,0.360768,-0.039101,3.954987,0.662464,0.709436,-3.566474,2
2,AAPL,320193,2019-10-31,10-K,0000320193-19-000119,2019,4,320193,32019319000119,AAPL_2019-10-31_10-K_0000320193-19-000119_FAMI...,...,0.009446,0.008896,0.023128,-1.456441,3.091214,4.107836,1.159828,0.733736,-4.628291,2245
3,AAPL,320193,2020-01-29,10-Q,0000320193-20-000010,2020,1,320193,32019320000010,AAPL_20200129_10-Q_000032019320000010.txt,...,0.013792,0.011713,0.029405,-0.158909,0.862831,4.374782,1.468833,1.343708,-3.852132,3
4,AAPL,320193,2020-05-01,10-Q,0000320193-20-000052,2020,2,320193,32019320000052,AAPL_20200501_10-Q_000032019320000052.txt,...,0.019945,0.019764,0.032845,0.076090,-1.717656,4.296605,0.738856,0.523305,-4.397591,4


Cell 7 — Keep only rows with valid embedding index

In [7]:
round3_df = round3_df.dropna(subset=["emb_idx"]).copy()
round3_df["emb_idx"] = round3_df["emb_idx"].astype(int)

print("Round 3 shape after keeping valid embedding rows:", round3_df.shape)

Round 3 shape after keeping valid embedding rows: (2181, 42)


In [8]:
split_date = pd.Timestamp("2023-01-01")

train_df = round3_df.loc[round3_df["filing_date"] < split_date].copy().reset_index(drop=True)
test_df = round3_df.loc[round3_df["filing_date"] >= split_date].copy().reset_index(drop=True)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train range:", train_df["filing_date"].min(), "to", train_df["filing_date"].max())
print("Test range:", test_df["filing_date"].min(), "to", test_df["filing_date"].max())

Train rows: 1394
Test rows: 787
Train range: 2019-04-02 00:00:00 to 2022-12-29 00:00:00
Test range: 2023-01-05 00:00:00 to 2024-12-13 00:00:00


In [9]:
X_train_emb = raw_emb[train_df["emb_idx"].to_numpy()]
X_test_emb = raw_emb[test_df["emb_idx"].to_numpy()]

print("Train embedding shape:", X_train_emb.shape)
print("Test embedding shape:", X_test_emb.shape)

Train embedding shape: (1394, 768)
Test embedding shape: (787, 768)


In [10]:
finbert_scaler = StandardScaler()
X_train_emb_scaled = finbert_scaler.fit_transform(X_train_emb)
X_test_emb_scaled = finbert_scaler.transform(X_test_emb)

print("Scaled train shape:", X_train_emb_scaled.shape)
print("Scaled test shape:", X_test_emb_scaled.shape)

Scaled train shape: (1394, 768)
Scaled test shape: (787, 768)


In [11]:
N_COMPONENTS = 15

finbert_pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_train_pca = finbert_pca.fit_transform(X_train_emb_scaled)
X_test_pca = finbert_pca.transform(X_test_emb_scaled)

print("Train PCA shape:", X_train_pca.shape)
print("Test PCA shape:", X_test_pca.shape)
print("Explained variance:", finbert_pca.explained_variance_ratio_.sum())

Train PCA shape: (1394, 15)
Test PCA shape: (787, 15)
Explained variance: 0.81092197


In [12]:
pca_cols = [f"finbert_pca_{i+1}" for i in range(N_COMPONENTS)]

train_pca_df = pd.DataFrame(X_train_pca, columns=pca_cols)
test_pca_df = pd.DataFrame(X_test_pca, columns=pca_cols)

train_df = pd.concat([train_df.reset_index(drop=True), train_pca_df], axis=1)
test_df = pd.concat([test_df.reset_index(drop=True), test_pca_df], axis=1)

print("Train final shape:", train_df.shape)
print("Test final shape:", test_df.shape)
train_df.head()

Train final shape: (1394, 57)
Test final shape: (787, 57)


,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,...,finbert_pca_6,finbert_pca_7,finbert_pca_8,finbert_pca_9,finbert_pca_10,finbert_pca_11,finbert_pca_12,finbert_pca_13,finbert_pca_14,finbert_pca_15
0,AAPL,320193,2019-05-01,10-Q,0000320193-19-000066,2019,2,320193,32019319000066,AAPL_20190501_10-Q_000032019319000066.txt,...,3.001102,1.276029,5.252378,2.698090,3.683895,2.020700,5.919838,-8.415214,4.769353,-3.897105
1,AAPL,320193,2019-07-31,10-Q,0000320193-19-000076,2019,3,320193,32019319000076,AAPL_20190731_10-Q_000032019319000076.txt,...,2.973329,1.289481,5.237220,2.703003,3.686063,2.039760,5.920076,-8.407431,4.765372,-3.907531
2,AAPL,320193,2019-10-31,10-K,0000320193-19-000119,2019,4,320193,32019319000119,AAPL_2019-10-31_10-K_0000320193-19-000119_FAMI...,...,-0.933556,1.913481,1.984130,0.827641,1.753077,2.450265,2.677766,0.272475,-1.494454,0.692461
3,AAPL,320193,2020-01-29,10-Q,0000320193-20-000010,2020,1,320193,32019320000010,AAPL_20200129_10-Q_000032019320000010.txt,...,-6.480239,1.436192,1.829646,0.598953,-0.441699,6.424584,2.416597,-0.747362,-2.251631,1.707531
4,AAPL,320193,2020-05-01,10-Q,0000320193-20-000052,2020,2,320193,32019320000052,AAPL_20200501_10-Q_000032019320000052.txt,...,-4.428791,0.668289,2.536207,-1.431850,-2.908610,6.019725,-1.600256,0.353318,-2.182583,-0.390294


In [13]:
train_df.to_csv("data/12_round3_finbert/train_round3_with_finbert_pca.csv", index=False)
test_df.to_csv("data/12_round3_finbert/test_round3_with_finbert_pca.csv", index=False)

np.save("data/12_round3_finbert/train_finbert_pca.npy", X_train_pca)
np.save("data/12_round3_finbert/test_finbert_pca.npy", X_test_pca)

with open("data/12_round3_finbert/finbert_scaler_train.pkl", "wb") as f:
    pickle.dump(finbert_scaler, f)

with open("data/12_round3_finbert/finbert_pca_train.pkl", "wb") as f:
    pickle.dump(finbert_pca, f)

print("Saved split-aware PCA files.")

Saved split-aware PCA files.


In [14]:
print("Train missing PCA values:", train_df[pca_cols].isna().sum().sum())
print("Test missing PCA values:", test_df[pca_cols].isna().sum().sum())
print("Explained variance:", finbert_pca.explained_variance_ratio_.sum())

Train missing PCA values: 0
Test missing PCA values: 0
Explained variance: 0.81092197


In [15]:
price_plus_context_features = [
    "past_return_5d",
    "past_return_10d",
    "past_return_20d",
    "abs_past_return_10d",
    "past_realized_vol_5d",
    "past_realized_vol_10d",
    "past_realized_vol_20d",
    "past_realized_vol_30d",
    "past_realized_vol_60d",
    "mean_abs_return_5d",
    "mean_abs_return_10d",
    "max_abs_return_10d",
    "return_skew_10d",
    "return_kurtosis_10d",
    "log_price_tminus1",
    "vol_ratio_5_20",
    "vol_ratio_10_60",
    "is_10k",
    "log_text_length_words"
]

categorical_features = ["quarter", "filing_month", "filing_year"]

lm_features = [
    "lm_negative",
    "lm_positive",
    "lm_uncertainty",
    "lm_net_sentiment"
]

finbert_features = pca_cols.copy()

target_col = "log_future_realized_vol_10d"

feature_sets = {
    "model1_price_only": price_plus_context_features + categorical_features,
    "model2_price_lm": price_plus_context_features + lm_features + categorical_features,
    "model3_price_lm_finbert": price_plus_context_features + lm_features + finbert_features + categorical_features
}

In [16]:
def evaluate_regression_both_scales(y_true_log, y_pred_log):
    # log scale metrics
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    mae_log = mean_absolute_error(y_true_log, y_pred_log)
    r2_log = r2_score(y_true_log, y_pred_log)

    # original scale metrics
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "RMSE_log": rmse_log,
        "MAE_log": mae_log,
        "R2_log": r2_log,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }

In [17]:
y_train = train_df[target_col].copy()
y_test = test_df[target_col].copy()

naive_pred = np.repeat(y_train.mean(), len(y_test))
naive_metrics = evaluate_regression_both_scales(y_test, naive_pred)

naive_results_df = pd.DataFrame([{
    "model": "NaiveMean",
    "dataset": "naive_mean",
    **naive_metrics
}])

naive_results_df

,model,dataset,RMSE_log,MAE_log,R2_log,RMSE,MAE,R2
0,NaiveMean,naive_mean,0.547993,0.444641,-0.25875,0.010087,0.006819,-0.035744


In [18]:
all_results = []

for dataset_name, cols in feature_sets.items():
    X_train = train_df[cols].copy()
    X_test = test_df[cols].copy()

    y_train = train_df[target_col].copy()
    y_test = test_df[target_col].copy()

    numeric_cols = [c for c in cols if c not in categorical_features]
    categorical_cols = categorical_features.copy()

    linear_preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), categorical_cols)
        ]
    )

    tree_preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), categorical_cols)
        ]
    )

    models = {
        "LinearRegression": Pipeline([
            ("prep", linear_preprocessor),
            ("model", LinearRegression())
        ]),
        "ElasticNet": Pipeline([
            ("prep", linear_preprocessor),
            ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42))
        ]),
        "RandomForest": Pipeline([
            ("prep", tree_preprocessor),
            ("model", RandomForestRegressor(
                n_estimators=400,
                max_depth=10,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            ))
        ]),
        "XGBoost": Pipeline([
            ("prep", tree_preprocessor),
            ("model", XGBRegressor(
                n_estimators=400,
                max_depth=4,
                learning_rate=0.03,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_alpha=0.0,
                reg_lambda=1.0,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            ))
        ])
    }

    for model_name, pipe in models.items():
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        metrics = evaluate_regression_both_scales(y_test, preds)
        metrics["model"] = model_name
        metrics["dataset"] = dataset_name
        all_results.append(metrics)

results_df = pd.DataFrame(all_results)
results_df = pd.concat([naive_results_df, results_df], ignore_index=True)
results_df = results_df.sort_values("RMSE").reset_index(drop=True)

results_df

,model,dataset,RMSE_log,MAE_log,R2_log,RMSE,MAE,R2
0,RandomForest,model3_price_lm_finbert,0.424610,0.334097,0.244264,0.008709,0.005195,0.227903
1,RandomForest,model1_price_only,0.425591,0.335995,0.240766,0.008749,0.005236,0.220727
2,RandomForest,model2_price_lm,0.425100,0.334164,0.242517,0.008763,0.005198,0.218346
3,XGBoost,model3_price_lm_finbert,0.433846,0.337091,0.211030,0.008770,0.005275,0.217129
4,XGBoost,model2_price_lm,0.436456,0.339010,0.201508,0.008890,0.005296,0.195412
5,XGBoost,model1_price_only,0.438021,0.342670,0.195773,0.008906,0.005351,0.192635
6,ElasticNet,model3_price_lm_finbert,0.470344,0.370341,0.072698,0.009425,0.005813,0.095643
7,LinearRegression,model3_price_lm_finbert,0.471210,0.371076,0.069280,0.009454,0.005834,0.090198
8,ElasticNet,model1_price_only,0.470734,0.371456,0.071159,0.009704,0.005845,0.041413
9,ElasticNet,model2_price_lm,0.470008,0.368822,0.074024,0.009737,0.005810,0.034959


In [19]:
results_df.to_csv("data/12_round3_finbert/round3_model_results.csv", index=False)
print("Saved Round 3 results.")

Saved Round 3 results.


In [20]:
best_dataset = "model3_price_lm_finbert"

numeric_cols = [c for c in feature_sets[best_dataset] if c not in categorical_features]
categorical_cols = categorical_features.copy()

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ]
)

rf_best = Pipeline([
    ("prep", tree_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=400,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

y_train_best = train_df[target_col].copy()
rf_best.fit(train_df[feature_sets[best_dataset]], y_train_best)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['past_return_5d',
                                                   'past_return_10d',
                                                   'past_return_20d',
                                                   'abs_past_return_10d',
                                                   'past_realized_vol_5d',
                                                   'past_realized_vol_10d',
                                                   'past_realized_vol_20d',
                                                   'past_realized_vol_30d',
                                                   'past_realized_vol_60d',
                                                   'mean_abs_return_5d',
                                                   'mean_ab...
                                                   'finbert_pca_4',
                                                   'finbert_pca_5',
                                                   'finbert_pca_6',
                                                   'finbert_pca_7', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['quarter', 'filing_month',
                                                   'filing_year'])])),
                ('model',
                 RandomForestRegressor(max_depth=10, min_samples_leaf=5,
                                       n_estimators=400, n_jobs=-1,
                                       random_state=42))])

In [23]:
prep = rf_best.named_steps["prep"]
model = rf_best.named_steps["model"]

feature_names = prep.get_feature_names_out()
importances = model.feature_importances_

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

feat_imp.head(30)

,feature,importance
10,num__mean_abs_return_10d,0.202427
8,num__past_realized_vol_60d,0.104015
7,num__past_realized_vol_30d,0.102758
6,num__past_realized_vol_20d,0.079008
9,num__mean_abs_return_5d,0.033228
43,cat__filing_month_2,0.029338
0,num__past_return_5d,0.029131
55,cat__filing_year_2020,0.025460
16,num__vol_ratio_10_60,0.022441
1,num__past_return_10d,0.020945


In [24]:
feat_imp.to_csv("data/12_round3_finbert/round3_feature_importance.csv", index=False)
print("Saved Round 3 feature importance.")

Saved Round 3 feature importance.


In [25]:
from google.colab import files

files.download("data/12_round3_finbert/round3_model_results.csv")
files.download("data/12_round3_finbert/round3_feature_importance.csv")
files.download("data/12_round3_finbert/train_round3_with_finbert_pca.csv")
files.download("data/12_round3_finbert/test_round3_with_finbert_pca.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>